In [5]:
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta
import random

In [ ]:
fake = Faker('es_ES')
np.random.seed(42)
random.seed(42)

# -----------------------
# PARÁMETROS GENERALES
# -----------------------
FECHA_INICIO = datetime(2024, 1, 1)
FECHA_FIN   = datetime(2025, 12, 31)

N_PACIENTES = 1500          # pacientes totales aprox
N_FISIOTERAPEUTAS = 5          # ya definidos en MySQL
N_TRATAMIENTOS = 6          # ya definidos en MySQL
SALAS = ['Sala 1', 'Sala 2', 'Sala 3', 'Sala entrenamiento']

# Porcentajes de origen de pacientes
ORIGEN = ['Privado', 'Recomendación', 'Redes sociales', 'Otro']
PROB_ORIGEN = [0.30, 0.35, 0.25, 0.10]  # suma 1.0

# Distribución nº de citas por paciente
# 1 cita, 2-4 citas, 5-8 citas
N_CITAS = [1, 2, 3, 6, 10, 20]
PROB_CITAS   = [0.3, 0.3, 0.2,  0.05]

# Probabilidades de estado de la cita
ESTADO = ['Realizada', 'Cancelada', 'No viene']
PROB_ESTADO = [0.88, 0.05, 0.07]

# Probabilidades de tipo de pago
PAGO = ['Sesión', 'Bono', 'Tarjeta regalo']
PROB_PAGO = [0.6, 0.35, 0.05]

TIPO_PAGO = ['Efectivo', 'Tarjeta']
PROB_TIPO_PAGO = [0.2, 0.8]

# Precio base por treatment_id (los mismos que en tu tabla treatments)
PRECIOS_TRATAMIENTOS = {
    1: 50.0,  # Fisioterapia
    2: 60.0,  # Suelo pélvico
    3: 55.0,  # Diatermia
    4: 65.0,  # Ejercicio terapéutico
    5: 50.0,  # Fisioterapia infantil
    6: 55.0   # Drenaje linfático manual
}

# Para bonos
SESIONES_BONO = [5]
DESCUENTO_BONO = 0.8   # 20% dto. vs precio suelto


# -----------------------
# FUNCIONES AUXILIARES
# -----------------------

def fecha_aleatoria(start, end):
    """Devuelve una fecha aleatoria entre start y end."""
    delta = end - start
    random_days = np.random.randint(0, delta.days + 1)
    return start + timedelta(days=int(random_days))

def hora_aleatoria(date):
    """
    Devuelve una hora de cita realista:
    L-V, entre 9-14 y 16-21, con preferencia por tardes.
    """
    # evitar fines de semana
    while date.weekday() >= 5:
        date += timedelta(days=1)

    # tramos horarios (tardes con más probabilidad)
    slots = [
        (9,  14, 0.4),  # mañana
        (16, 21, 0.6)   # tarde
    ]
    r = np.random.rand()
    if r < slots[0][2]:
        start_h, end_h = slots[0][0], slots[0][1]
    else:
        start_h, end_h = slots[1][0], slots[1][1]

    hour = np.random.randint(start_h, end_h)
    return datetime(date.year, date.month, date.day, hour)

def choose_citas_count():
    """Elige cuántas citas tendrá un paciente (1, 2-4, 5-8 aprox)."""
    bucket = np.random.choice(CITAS_BUCKETS, p=CITAS_PROBS)
    if bucket == 1:
        return 1
    elif bucket == 3:
        return np.random.randint(2, 5)   # 2-4
    else:
        return np.random.randint(5, 9)   # 5-8

def generate_patients(n_patients):
    patients = []
    for pid in range(1, n_patients + 1):
        gender = np.random.choice(['F', 'M', 'O'], p=[0.5, 0.48, 0.02])
        profile = fake.simple_profile(sex='F' if gender == 'F' else 'M')
        first_name = profile['name'].split()[0]
        last_name  = ' '.join(profile['name'].split()[1:]) if len(profile['name'].split()) > 1 else fake.last_name()

        # edad entre 18 y 80
        age = np.random.randint(18, 81)
        birth_year = datetime.now().year - age
        birth_date = datetime(birth_year, np.random.randint(1, 13), np.random.randint(1, 28))

        registration_date = random_date(START_DATE, END_DATE)
        origin = np.random.choice(ORIGINS, p=ORIGIN_PROBS)

        patients.append({
            'patient_id': pid,
            'first_name': first_name,
            'last_name': last_name,
            'gender': gender,
            'birth_date': birth_date.date(),
            'origin': origin,
            'registration_date': registration_date.date()
        })
    return pd.DataFrame(patients)

def generate_bonos_for_patients(patients_df):
    """
    Genera bonos para ~15-20% de los pacientes.
    Bonos solo para treatment_id=2 (Seguimiento estándar).
    """
    bonos = []
    bono_id = 1
    # Seleccionamos un subconjunto de pacientes
    patients_with_bono = patients_df.sample(frac=0.18, random_state=42)

    for _, row in patients_with_bono.iterrows():
        n_bonos_patient = np.random.choice([1, 2], p=[0.8, 0.2])

        # cada paciente puede tener 1 o 2 bonos a lo largo del tiempo
        for _ in range(n_bonos_patient):
            sessions_total = np.random.choice(BONO_SESSION_OPTIONS, p=BONO_SESSION_PROBS)
            base_price = TREATMENT_PRICES[2]
            price_per_session = round(base_price * BONO_DISCOUNT_FACTOR, 2)
            total_amount = round(price_per_session * sessions_total, 2)

            purchase_date = random_date(
                max(START_DATE, row['registration_date']),
                END_DATE - timedelta(days=60)
            )
            expiry_date = purchase_date + timedelta(days=np.random.randint(90, 270))  # 3-9 meses

            bonos.append({
                'bono_id': bono_id,
                'patient_id': row['patient_id'],
                'treatment_id': 2,  # Seguimiento estándar
                'sessions_total': sessions_total,
                'sessions_used': 0,       # se actualizará al generar citas
                'total_amount_eur': total_amount,
                'price_per_session': price_per_session,
                'purchase_date': purchase_date.date(),
                'expiry_date': expiry_date.date(),
                'status': 'Activo'        # luego ajustaremos a Expirado/Agotado
            })
            bono_id += 1

    return pd.DataFrame(bonos)

def generate_appointments_and_update_bonos(patients_df, bonos_df):
    appointments = []
    appointment_id = 1

    # Índice rápido de bonos por paciente
    bonos_by_patient = {pid: [] for pid in patients_df['patient_id']}
    for idx, row in bonos_df.iterrows():
        bonos_by_patient[row['patient_id']].append(row)

    for _, patient in patients_df.iterrows():
        n_citas = choose_citas_count()

        # primera cita cerca de la fecha de registro
        first_date = random_date(
            max(START_DATE, patient['registration_date']),
            END_DATE - timedelta(days=1)
        )
        # aseguramos que no sea fin de semana
        current_date = first_date

        # si el paciente tiene bonos, los usaremos en parte de las citas
        patient_bonos = bonos_by_patient.get(patient['patient_id'], [])

        for i in range(n_citas):
            # definimos tipo de pago según probabilidades, pero si hay bono disponible,
            # aumentamos la probabilidad de 'Bono'
            base_probs = PAYMENT_PROBS.copy()
            if patient_bonos:
                # aumentamos algo la probabilidad de que sea bono
                base_probs = [0.25, 0.35, 0.30, 0.10]  # algo más de 'Bono'

            payment_type = np.random.choice(PAYMENT_TYPES, p=base_probs)

            # elegimos tratamiento: si pago con bono, treatment_id=2
            if payment_type == 'Bono' and patient_bonos:
                treatment_id = 2
            else:
                # distribución simple: más probabilidad de seguimiento
                treatment_id = np.random.choice(
                    [1, 2, 3, 4, 5],
                    p=[0.10, 0.55, 0.15, 0.10, 0.10]
                )

            # generar fecha/hora realista
            appt_datetime = random_datetime_in_working_hours(current_date)

            status = np.random.choice(STATUS, p=STATUS_PROBS)

            # precio base
            base_price = TREATMENT_PRICES[treatment_id]
            # ligera variación +/- 5%
            variation = np.random.uniform(0.95, 1.05)
            amount = round(base_price * variation, 2)

            # si se paga con bono, buscamos un bono con sesiones disponibles
            if payment_type == 'Bono' and patient_bonos:
                usable_bonos = [b for b in patient_bonos if b['sessions_used'] < b['sessions_total']]
                if usable_bonos:
                    bono = random.choice(usable_bonos)
                    # usar precio por sesión del bono
                    amount = bono['price_per_session']

                    # si la cita se realiza, consumimos una sesión
                    if status == 'Realizada':
                        bono['sessions_used'] += 1

            # tipo de pago 'Seguro' o 'Mutua' podría facturar un poco menos
            if payment_type in ['Seguro', 'Mutua']:
                amount = round(amount * np.random.uniform(0.7, 0.9), 2)

            therapist_id = np.random.randint(1, N_THERAPISTS + 1)
            room = random.choice(ROOMS)

            created_at = appt_datetime - timedelta(days=np.random.randint(1, 15))

            appointments.append({
                'appointment_id': appointment_id,
                'patient_id': patient['patient_id'],
                'therapist_id': therapist_id,
                'treatment_id': treatment_id,
                'appointment_datetime': appt_datetime,
                'status': status,
                'payment_type': payment_type,
                'amount_charged_eur': amount,
                'room': room,
                'created_at': created_at
            })
            appointment_id += 1

            # siguiente cita 3-14 días después
            current_date = current_date + timedelta(days=np.random.randint(3, 15))

    # actualizar estado de bonos (Agotado / Expirado / Activo)
    for idx, row in bonos_df.iterrows():
        used = row['sessions_used']
        total = row['sessions_total']
        if used >= total:
            bonos_df.at[idx, 'status'] = 'Agotado'
        else:
            # si la fecha actual es posterior a expiry, lo marcamos Expirado
            if row['expiry_date'] < END_DATE.date():
                bonos_df.at[idx, 'status'] = 'Expirado'
            else:
                bonos_df.at[idx, 'status'] = 'Activo'

    return pd.DataFrame(appointments), bonos_df


# -----------------------
# GENERACIÓN DE DATOS
# -----------------------

patients_df = generate_patients(N_PATIENTS)
bonos_df = generate_bonos_for_patients(patients_df)
appointments_df, bonos_df = generate_appointments_and_update_bonos(patients_df, bonos_df)

# Ordenar por fecha para que quede más limpio
appointments_df = appointments_df.sort_values(by='appointment_datetime').reset_index(drop=True)

# -----------------------
# EXPORTAR A CSV
# -----------------------

patients_df.to_csv('patients.csv', index=False)
appointments_df.to_csv('appointments.csv', index=False)
bonos_df.to_csv('bonos.csv', index=False)

print('CSV generados: patients.csv, appointments.csv, bonos.csv')
